# TP5 — Partie A : Agentic AI avec LangChain

Un agent = **MIND** (le LLM qui raisonne) + **BODY** (les tools qui agissent), piloté par le pattern **ReAct** :
*Raisonnement → Action (appel d'un tool) → Observation*, en boucle jusqu'à la réponse finale.

Plan (dans l'ordre de l'énoncé) :
1. Créer un simple agent
2. Ajouter une mémoire à l'agent
3. Ajouter des tools à l'agent
4. Utiliser des tools prédéfinis : DuckDuckGo, Tavily, PythonREPLTool
5. Ajouter des middlewares : `dynamic_model`, `dynamic_prompt`, gestion des erreurs des tools, Guardrails, Human In The Loop
6. Créer un chatbot basé sur un RAG agentique

**LLM utilisé :** `qwen3-vl:8b` servi localement par **Ollama** (aucune clé API requise), via `ChatOllama`.
C'est un modèle *thinking* : il raisonne toujours avant de répondre (d'où des temps de réponse de l'ordre de la minute).

In [1]:
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama

load_dotenv(override=True)

MODELE = "qwen3-vl:8b"
# reasoning=False : le raisonnement interne du modèle (thinking) n'est pas renvoyé dans la réponse
# num_ctx : fenêtre de contexte d'Ollama (4096 par défaut, trop court pour les résultats de recherche web)
llm = ChatOllama(model=MODELE, temperature=0, reasoning=False, num_ctx=16384)


def trace(resp, debut=0):
    """Affiche le cycle ReAct (à partir du message `debut`) : appels de tools, observations, réponse finale."""
    for m in resp["messages"][debut:]:
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                print(f"🛠  Action      : {tc['name']}({tc['args']})")
        elif isinstance(m, ToolMessage):
            print(f"👁  Observation : {str(m.content)[:200]}")
    display(Markdown("**Réponse :** " + resp["messages"][-1].content))

## 1. Simple agent

`create_agent` construit un agent ReAct (un graphe LangGraph prêt à l'emploi) à partir d'un modèle, d'une liste de tools et d'un system prompt.

In [2]:
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant",
)

response = agent.invoke({"messages": [{"role": "user", "content": "My name is Mohamed"}]})
display(Markdown(response["messages"][-1].content))

Hello, Mohamed! 😊 It's great to meet you. How can I assist you today? Feel free to ask me anything!

In [3]:
response = agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]})
display(Markdown(response["messages"][-1].content))

I don't have access to your personal information, so I don't know your name. How can I assist you today? 😊

➡️ Sans mémoire, chaque `invoke` est indépendant : l'agent **ne connaît pas** le nom donné au tour précédent.

### Configuration et contrôle du modèle
On peut instancier directement la classe du fournisseur (`ChatOllama` ici, `ChatOpenAI` dans le cours) pour régler température, longueur maximale, etc. `debug=True` affiche chaque étape du graphe.

In [4]:
llm_config = ChatOllama(model=MODELE, temperature=0.1, num_predict=1000, reasoning=False, num_ctx=16384)

agent = create_agent(model=llm_config, tools=[], system_prompt="You are a helpful assistant", debug=True)
response = agent.invoke({"messages": [HumanMessage("La ville la plus grande au Maroc")]})
display(Markdown(response["messages"][-1].content))

[values] {'messages': [HumanMessage(content='La ville la plus grande au Maroc', additional_kwargs={}, response_metadata={}, id='0de6de12-374b-4b8b-b740-9c26ea7e5efd')]}


[updates] {'model': {'messages': [AIMessage(content="La ville la plus grande au Maroc est **Casablanca**. Elle est située sur la côte atlantique et est le centre économique, commercial et industriel principal du pays. Selon les données démographiques récentes, Casablanca compte environ **3,5 millions d'habitants** (zone urbaine), ce qui la place en tête des villes marocaines en termes de population. \n\nBien que **Rabat** soit la capitale du Maroc, elle est moins peuplée (environ 1,3 million d'habitants). D'autres villes importantes comme Marrakech ou Fès sont également connues pour leur richesse historique et culturelle, mais elles ne dépassent pas Casablanca en taille démographique. \n\nCasablanca est également un important centre portuaire et un hub international, attirant des investissements et des échanges commerciaux significatifs.", additional_kwargs={}, response_metadata={'model': 'qwen3-vl:8b', 'created_at': '2026-11-23T02:59:31.527286Z', 'done': True, 'done_reason': 'stop', '

La ville la plus grande au Maroc est **Casablanca**. Elle est située sur la côte atlantique et est le centre économique, commercial et industriel principal du pays. Selon les données démographiques récentes, Casablanca compte environ **3,5 millions d'habitants** (zone urbaine), ce qui la place en tête des villes marocaines en termes de population. 

Bien que **Rabat** soit la capitale du Maroc, elle est moins peuplée (environ 1,3 million d'habitants). D'autres villes importantes comme Marrakech ou Fès sont également connues pour leur richesse historique et culturelle, mais elles ne dépassent pas Casablanca en taille démographique. 

Casablanca est également un important centre portuaire et un hub international, attirant des investissements et des échanges commerciaux significatifs.

## 2. Mémoire de l'agent

On ajoute un **checkpointer** (`InMemorySaver`) : l'état (l'historique des messages) est sauvegardé par `thread_id`.
Deux appels avec le même `thread_id` partagent la même conversation.

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "1"}}
res = agent.invoke({"messages": [HumanMessage("My Name is Mohamed")]}, config=config)
print(res["messages"][-1].content)

Hello, Mohamed! 😊 It's great to meet you. How can I assist you today? Feel free to ask me anything!


In [6]:
res = agent.invoke({"messages": [HumanMessage("What is my name?")]}, config=config)
print(res["messages"][-1].content)

Your name is **Mohamed**! 😊 I remember you mentioned it earlier. How can I assist you today?


In [7]:
# Un autre thread = une autre conversation : l'agent ne connaît pas le nom
res = agent.invoke({"messages": [HumanMessage("What is my name?")]}, config={"configurable": {"thread_id": "2"}})
print(res["messages"][-1].content)

I don't have access to your personal information, so I don't know your name. How can I assist you today? 😊


In [8]:
# Contenu de la mémoire du thread 1
for m in agent.get_state(config).values["messages"]:
    m.pretty_print()

================================ Human Message =================================

My Name is Mohamed
================================== Ai Message ==================================

Hello, Mohamed! 😊 It's great to meet you. How can I assist you today? Feel free to ask me anything!
================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

Your name is **Mohamed**! 😊 I remember you mentioned it earlier. How can I assist you today?


## 3. Ajouter des tools à l'agent

Un tool est une fonction Python décorée par `@tool` : sa **docstring** sert de description au LLM, qui décide seul quand l'appeler et avec quels arguments.

In [9]:
from langchain.tools import tool


@tool
def search(query: str) -> dict:
    """Search for news."""
    print(f"Search tool is called for {query}")
    return {"query": query, "news": ["Le temps est très glacial", "les conditions météo sont très délicates"]}


@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    print(f"Weather tool is called for {location}")
    return f"Weather in {location}: Sunny, 32°C"


@tool
def get_employee_info(name: str) -> dict:
    """Get information about the given employee name (salary, job)."""
    print(f"get_employee_info tool is called for {name}")
    return {"name": name, "salary": 12000, "job": "Developper"}


agent = create_agent(model=llm, tools=[search, get_weather, get_employee_info],
                     system_prompt="Answer the question using these tools")

In [10]:
trace(agent.invoke({"messages": [HumanMessage("What is the weather in Marrakech")]}))

Weather tool is called for Marrakech


🛠  Action      : get_weather({'location': 'Marrakech'})
👁  Observation : Weather in Marrakech: Sunny, 32°C


**Réponse :** The current weather in Marrakech is **Sunny** with a temperature of **32°C**.

In [11]:
trace(agent.invoke({"messages": [HumanMessage("What are the news?")]}))

Search tool is called for news


🛠  Action      : search({'query': 'news'})
👁  Observation : {"query": "news", "news": ["Le temps est très glacial", "les conditions météo sont très délicates"]}


**Réponse :** The latest news includes:  
- The weather is very cold.  
- Weather conditions are very challenging.

In [12]:
trace(agent.invoke({"messages": [HumanMessage("Quel est le salaire et le job de Mohamed")]}))

get_employee_info tool is called for Mohamed


🛠  Action      : get_employee_info({'name': 'Mohamed'})
👁  Observation : {"name": "Mohamed", "salary": 12000, "job": "Developper"}


**Réponse :** Le salaire de Mohamed est de 12000 et son poste est Developper.

## 4. Tools prédéfinis

### 4.1 DuckDuckGo
LangChain fournit un tool de recherche web DuckDuckGo prêt à l'emploi (`DuckDuckGoSearchResults`, basé sur la librairie `ddgs`), sans clé API.

In [13]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

import time


class DuckDuckGoAvecRetry(DuckDuckGoSearchResults):
    """Tool DuckDuckGo prédéfini + nouvelles tentatives (le service limite souvent le débit)."""

    def _run(self, query: str, run_manager=None) -> str:
        for essai in range(1, 6):
            try:
                return super()._run(query, run_manager)
            except Exception as e:
                print(f"tentative {essai} échouée : {e}")
                erreur = e
                time.sleep(2)
        return f"Recherche impossible : {erreur}"


# region "us-en" : la valeur par défaut "wt-wt" provoque une erreur DNS dans ddgs
ddg_tool = DuckDuckGoAvecRetry(api_wrapper=DuckDuckGoSearchAPIWrapper(region="us-en", backend="auto", max_results=5))
print(ddg_tool.name, "→", ddg_tool.description[:120], "...")

agent = create_agent(
    model=llm,
    tools=[ddg_tool, get_employee_info, get_weather],
    system_prompt="You are a helpful assistant. Use the web search tool for any question about the web.",
)
trace(agent.invoke({"messages": [HumanMessage("Search for python tutorials")]}))

/var/folders/ms/63vgmytn49z1hq2nlcc_fx840000gn/T/ipykernel_99377/1980579615.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


duckduckgo_results_json → A wrapper around Duck Duck Go Search. Useful for when you need to answer questions about current events. Input should be ...


tentative 1 échouée : No results found.


🛠  Action      : duckduckgo_results_json({'query': 'python tutorials'})
👁  Observation : snippet: Learn Python in bite-sized quests with XP and streaks. Your progress grows for each new adventure! Free on iOS and Android. With our "Try it Yourself" editor, you can edit Python code and vie


**Réponse :** Here are some popular Python tutorial resources based on your search:

1. **W3Schools Python Tutorial**  
   Interactive learning with hands-on code editor. Features "Try it Yourself" exercises to practice immediately.  
   [🔗 Visit W3Schools](https://www.w3schools.com/python/)

2. **Official Python Documentation Tutorial**  
   The authoritative guide from Python's creators. Covers fundamentals to advanced topics with clear examples.  
   [🔗 Official Tutorial](https://docs.python.org/3/tutorial/index.html)

3. **GeeksforGeeks Python Tutorial**  
   Comprehensive guide from basics to advanced concepts, with practical examples and problem-solving approaches.  
   [🔗 GeeksforGeeks Tutorial](https://www.geeksforgeeks.org/python/python-programming-language-tutorial/)

4. **Tutorialspoint Python Tutorial**  
   Structured learning path covering core concepts, syntax, and real-world applications.  
   [🔗 Tutorialspoint](https://www.tutorialspoint.com/python/index.htm)

All these resources are beginner-friendly and widely recommended for learning Python. Would you like more details about any specific tutorial? 😊

### 4.2 Tavily
Moteur de recherche conçu pour les agents (résultats déjà nettoyés). La clé `TAVILY_API_KEY` est lue dans le fichier `.env` (non versionné).

In [14]:
import os
from datetime import datetime
from langchain_tavily import TavilySearch

assert os.getenv("TAVILY_API_KEY"), "Ajouter TAVILY_API_KEY dans le fichier .env"

tavily_search_tool = TavilySearch(max_results=5, search_depth="advanced")

agent = create_agent(
    model=llm,
    tools=[tavily_search_tool],
    system_prompt=f"""You are a helpful assistant that searches the web for information using the search tool.
Today's date is {datetime.now().strftime('%Y-%m-%d')}. Answer in French.""",
)
trace(agent.invoke({"messages": [HumanMessage("Quel temps fait-il aujourd'hui à Casablanca ?")]}))

🛠  Action      : tavily_search({'query': "Quel temps fait-il aujourd'hui à Casablanca ?", 'time_range': 'day'})
👁  Observation : {"query": "Quel temps fait-il aujourd'hui à Casablanca ?", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.meteohorizon.com/africa/morocco/casablanca-settat


**Réponse :** Aujourd'hui à Casablanca, il fait nuageux avec une température de 16°C, une humidité de 74% et un vent de 12 km/h. Il y a une faible probabilité de pluie (15%). La température maximale de la journée est prévue à 21,5°C, avec des brises légères et un ciel partiellement couvert.

### 4.3 PythonREPLTool
Un shell Python : l'agent **génère** du code puis l'**exécute** pour obtenir un résultat exact.
Pour écrire le fichier `doc.txt`, on lui ajoute un second tool prédéfini, `WriteFileTool` (évite au LLM d'échapper du code dans une chaîne Python).
⚠️ Ce tool exécute du code arbitraire : à réserver à un environnement de confiance.

In [15]:
from langchain_experimental.utilities import PythonREPL

python_repl = PythonREPL()
python_repl.run('print(f"la somme de 5 et 9 est {5+9}")')

/var/folders/ms/63vgmytn49z1hq2nlcc_fx840000gn/T/ipykernel_95036/754029842.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.utilities import PythonREPL
Python REPL can execute arbitrary code. Use with caution.


'la somme de 5 et 9 est 14\n'

In [16]:
from langchain_experimental.tools import PythonREPLTool
from langchain_community.tools import WriteFileTool

agent_repl = create_agent(
    model=llm,
    tools=[PythonREPLTool(), WriteFileTool(root_dir=".")],
    system_prompt="""Generate the python code.
Use the Python REPL tool to execute the generated code (always print the results).
Then use the write_file tool to save the generated python code and the execution result in the file doc.txt.
Then give the final answer in French: the sorted lists and the result.""",
)
trace(agent_repl.invoke({"messages": [HumanMessage(
    "Je veux trier deux listes [6,5,8,3,2] et [65,15,8,13,2] et puis faire la somme des deux listes triées")]}))

🛠  Action      : Python_REPL({'query': 'list1 = [6,5,8,3,2]\nlist2 = [65,15,8,13,2]\nsorted_list1 = sorted(list1)\nsorted_list2 = sorted(list2)\nsum_list = [a + b for a, b in zip(sorted_list1, sorted_list2)]\nprint("Sorted list 1:", sorted_list1)\nprint("Sorted list 2:", sorted_list2)\nprint("Sum:", sum_list)'})
🛠  Action      : write_file({'file_path': 'doc.txt', 'text': 'list1 = [6,5,8,3,2]\nlist2 = [65,15,8,13,2]\nsorted_list1 = sorted(list1)\nsorted_list2 = sorted(list2)\nsum_list = [a + b for a, b in zip(sorted_list1, sorted_list2)]\nprint("Sorted list 1:", sorted_list1)\nprint("Sorted list 2:", sorted_list2)\nprint("Sum:", sum_list)\n\nSorted list 1: [2, 3, 5, 6, 8]\nSorted list 2: [2, 8, 13, 15, 65]\nSum: [4, 11, 18, 21, 73]', 'append': False})
🛠  Action      : Python_REPL({'query': 'list1 = [6,5,8,3,2]\nlist2 = [65,15,8,13,2]\nsorted_list1 = sorted(list1)\nsorted_list2 = sorted(list2)\nsum_list = [a + b for a, b in zip(sorted_list1, sorted_list2)]\nprint("Sorted list 1:", sorte

**Réponse :** Les listes triées sont [2, 3, 5, 6, 8] et [2, 8, 13, 15, 65]. La somme des listes triées est [4, 11, 18, 21, 73].

In [17]:
# Vérification : le fichier écrit par l'agent
print(open("doc.txt").read())

list1 = [6,5,8,3,2]
list2 = [65,15,8,13,2]
sorted_list1 = sorted(list1)
sorted_list2 = sorted(list2)
sum_list = [a + b for a, b in zip(sorted_list1, sorted_list2)]
print("Sorted list 1:", sorted_list1)
print("Sorted list 2:", sorted_list2)
print("Sum:", sum_list)

Sorted list 1: [2, 3, 5, 6, 8]
Sorted list 2: [2, 8, 13, 15, 65]
Sum: [4, 11, 18, 21, 73]


## 5. Middlewares

Un middleware intercepte l'exécution de l'agent à des points stratégiques : avant/après l'agent, avant/après le modèle, **autour** de l'appel au modèle (`wrap_model_call`) ou d'un tool (`wrap_tool_call`).

### 5.1 `dynamic_model` — choisir le modèle à l'exécution
Selon le contexte d'exécution (`env`), le middleware remplace le modèle de la requête par un modèle **basique** ou **avancé** (`request.override(model=...)`).

> Dans le cours : `gpt-4o-mini` (basique) / `gpt-4o` (avancé). Ici, un seul modèle local est disponible : les deux sont deux **configurations** de `qwen3-vl:8b` (mêmes poids) — basique : raisonnement masqué ; avancé : raisonnement renvoyé (`reasoning=True`) pour être journalisé. Ce qui est démontré est le **mécanisme de routage** ; en production on y branche deux modèles distincts.

In [18]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

basic_model = ChatOllama(model=MODELE, temperature=0, reasoning=False, num_ctx=16384)
advanced_model = ChatOllama(model=MODELE, temperature=0, reasoning=True, num_ctx=16384)


@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose the model based on the runtime context."""
    env = (request.runtime.context or {}).get("env", "test")
    if env == "prod":
        model, nom = advanced_model, "advanced (raisonnement renvoyé)"
    else:
        model, nom = basic_model, "basic (raisonnement masqué)"
    print(f"env={env} → modèle sélectionné : {nom}")
    return handler(request.override(model=model))


agent = create_agent(model=basic_model, tools=[], middleware=[dynamic_model_selection])
question = {"messages": [HumanMessage("Combien font 17 × 23 ? Réponds en une phrase.")]}

In [19]:
r = agent.invoke(question, context={"env": "test"})
m = r["messages"][-1]
print("raisonnement renvoyé :", "reasoning_content" in m.additional_kwargs)
print("réponse :", m.content)

env=test → modèle sélectionné : basic (raisonnement masqué)


raisonnement renvoyé : False
réponse : 17 × 23 = 391.


In [20]:
r = agent.invoke(question, context={"env": "prod"})
m = r["messages"][-1]
print("raisonnement renvoyé :", m.additional_kwargs.get("reasoning_content", "")[:300], "...")
print("réponse :", m.content)

env=prod → modèle sélectionné : advanced (raisonnement renvoyé)


raisonnement renvoyé : Okay, let's see. I need to calculate 17 multiplied by 23. Hmm, how do I do this? Maybe I can break it down. Let me think.

First, I remember that multiplying two numbers can sometimes be easier if I split them into parts. Like, 17 is 10 + 7 and 23 is 20 + 3. Wait, maybe using the distributive proper ...
réponse : 17 × 23 = 391.


### 5.2 `dynamic_prompt` — adapter le system prompt
Le system prompt est généré à chaque appel selon le rôle de l'utilisateur passé dans le `context`.

In [21]:
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt


class Context(TypedDict):
    user_role: str


@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant. Answer in French in at most 5 lines."
    if user_role == "expert":
        prompt = f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        prompt = f"{base_prompt} Explain concepts simply and avoid jargon."
    else:
        prompt = base_prompt
    print(f"user_role={user_role} → system prompt : {prompt}")
    return prompt


agent = create_agent(model=llm, tools=[], middleware=[user_role_prompt], context_schema=Context)

In [22]:
r = agent.invoke({"messages": [{"role": "user", "content": "Explain machine learning"}]}, context={"user_role": "expert"})
display(Markdown(r["messages"][-1].content))

user_role=expert → system prompt : You are a helpful assistant. Answer in French in at most 5 lines. Provide detailed technical responses.


L'apprentissage automatique (ML) est une discipline de l'intelligence artificielle où les modèles apprennent à partir de données via des algorithmes statistiques, sans programmation explicite. Il utilise des données d'entraînement pour ajuster des paramètres (ex: poids dans les réseaux de neurones) et minimiser des fonctions de perte. Les méthodes supervisées (régression, classification) nécessitent des étiquettes, tandis que les non supervisées (clustering, réduction de dimensionnalité) explorent les structures sans étiquettes. Les défis incluent la gestion des biais, l'overfitting et l'interprétabilité des modèles complexes.

In [23]:
r = agent.invoke({"messages": [{"role": "user", "content": "Explain machine learning"}]}, context={"user_role": "beginner"})
display(Markdown(r["messages"][-1].content))

user_role=beginner → system prompt : You are a helpful assistant. Answer in French in at most 5 lines. Explain concepts simply and avoid jargon.


Machine learning est une branche de l'intelligence artificielle qui permet aux ordinateurs d'apprendre automatiquement à partir de données. Il utilise des exemples pour s'améliorer sans instructions précises. Par exemple, pour des recommandations ou détecter du spam. L'ordinateur identifie des motifs et prend des décisions. Cela rend les systèmes plus intelligents avec l'expérience.

### 5.3 Gestion des erreurs des tools (`wrap_tool_call`)
Sans middleware, une exception levée dans un tool interrompt l'agent. Avec `wrap_tool_call`, l'erreur est transformée en `ToolMessage` : le LLM l'**observe** et peut corriger son action.

In [24]:
from langchain.agents.middleware import wrap_tool_call


@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    print(f"divide tool is called with a={a}, b={b}")
    return a / b


@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        print(f"⚠️  Erreur interceptée dans {request.tool_call['name']} : {e!r}")
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({e})",
            tool_call_id=request.tool_call["id"],
        )


agent = create_agent(model=llm, tools=[divide], middleware=[handle_tool_errors],
                     system_prompt="Use the tools to compute. Answer in French.")
trace(agent.invoke({"messages": [HumanMessage("Divise 10 par 0, puis divise 10 par 4")]}))

divide tool is called with a=10.0, b=4.0divide tool is called with a=10.0, b=0.0
⚠️  Erreur interceptée dans divide : ZeroDivisionError('float division by zero')

divide tool is called with a=10.0, b=0.0
⚠️  Erreur interceptée dans divide : ZeroDivisionError('float division by zero')
divide tool is called with a=10.0, b=4.0


🛠  Action      : divide({'a': 10, 'b': 0})
🛠  Action      : divide({'a': 10, 'b': 4})
🛠  Action      : divide({'a': 10, 'b': 0})
🛠  Action      : divide({'a': 10, 'b': 4})
👁  Observation : Tool error: Please check your input and try again. (float division by zero)
👁  Observation : 2.5
👁  Observation : Tool error: Please check your input and try again. (float division by zero)
👁  Observation : 2.5


**Réponse :** La division de 10 par 0 est impossible (division par zéro). La division de 10 par 4 est 2,5.

### 5.4 Guardrails (garde-fous)

Deux approches complémentaires :
- **déterministes** (regex, mots-clés) : rapides, prévisibles ;
- **fondées sur un modèle** (un LLM juge le contenu) : compréhension sémantique, plus coûteuses.

#### a) Garde-fou intégré : détection des PII (`PIIMiddleware`)
| Stratégie | Effet |
|---|---|
| `redact` | remplace par `[REDACTED_EMAIL]` |
| `mask` | masque partiellement `****-****-****-5100` |
| `block` | lève une exception |

In [25]:
from langchain.agents.middleware import PIIMiddleware


@tool
def card_tool(credit_card: str) -> str:
    """Process the payment using the credit card"""
    print(f"Process the payment using the credit card : {credit_card}")
    return f"The payment has been processed successfully with {credit_card}"


@tool
def send_email(email_address: str) -> str:
    """Send an email to the email_address"""
    print(f"Sending the email to : {email_address}")
    return f"Email sent successfully to {email_address}"


agent = create_agent(
    model=llm,
    tools=[card_tool, send_email],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("api_key", detector=r"sk-[a-zA-Z0-9]{32}", strategy="block", apply_to_input=True),
    ],
)

result = agent.invoke({"messages": [{"role": "user",
    "content": "send email to med@gmail.com and process the card 5105-1051-0510-5100."}]})
print("\n", result["messages"][-1].content)

Sending the email to : [REDACTED_EMAIL]
Process the payment using the credit card : ****-****-****-5100



 The email has been successfully sent to [REDACTED_EMAIL], and the payment using the card ending in ****-****-****-5100 has been processed. Both actions are complete. Let me know if you need further assistance!


In [26]:
# Stratégie "block" : une clé API dans le message lève une exception avant tout appel au modèle
try:
    agent.invoke({"messages": [{"role": "user",
        "content": "Use my key sk-abcdefghijklmnopqrstuvwxyz123456 to pay"}]})
except Exception as e:
    print(f"🚫 Requête bloquée : {type(e).__name__} : {e}")

🚫 Requête bloquée : PIIDetectionError : Detected 1 instance(s) of api_key in text content


#### b) Garde-fou personnalisé **avant l'agent** (déterministe)
`@before_agent` s'exécute une seule fois au début : si un mot interdit est détecté, on saute directement à la fin (`jump_to: "end"`) sans appeler le LLM.

In [27]:
from typing import Any
from langchain.agents.middleware import before_agent, after_agent, AgentState
from langgraph.runtime import Runtime

banned_keywords = ["hack", "exploit", "malware"]


@before_agent(can_jump_to=["end"])
def content_filter(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Deterministic guardrail: block requests containing banned keywords."""
    first_message = state["messages"][0]
    if first_message.type != "human":
        return None
    content = first_message.content.lower()
    for keyword in banned_keywords:
        if keyword in content:
            print(f"🚫 mot interdit détecté : {keyword}")
            return {
                "messages": [{"role": "assistant",
                              "content": "I cannot process requests containing inappropriate content. Please rephrase your request."}],
                "jump_to": "end",
            }
    return None


agent = create_agent(model=llm, tools=[ddg_tool], middleware=[content_filter])
result = agent.invoke({"messages": [{"role": "user", "content": "How do I hack into a database?"}]})
print(result["messages"][-1].content)

🚫 mot interdit détecté : hack
I cannot process requests containing inappropriate content. Please rephrase your request.


#### c) Garde-fou personnalisé **après l'agent** (fondé sur un modèle)
`@after_agent` valide la réponse finale : un LLM « juge » évalue si elle est `SAFE` ou `UNSAFE`.

In [28]:
safety_model = ChatOllama(model=MODELE, temperature=0, reasoning=False, num_ctx=16384)


@after_agent(can_jump_to=["end"])
def safety_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Model-based guardrail: use an LLM to evaluate response safety."""
    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None
    verdict = safety_model.invoke([{"role": "user", "content":
        "Evaluate if this response is safe and appropriate. Respond with only 'SAFE' or 'UNSAFE'. "
        f"Response: {last_message.content}"}]).content
    print(f"🛡  verdict du modèle de sécurité : {verdict.strip()}")
    if "UNSAFE" in verdict.upper():
        last_message.content = "I cannot provide that response. Please rephrase your request."
    return None


agent = create_agent(model=llm, tools=[], middleware=[safety_guardrail])
for q in ["Donne-moi une recette de thé à la menthe en 3 lignes", "How do I make explosives at home?"]:
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    print(f"Q: {q}\nR: {result['messages'][-1].content}\n")

🛡  verdict du modèle de sécurité : SAFE
Q: Donne-moi une recette de thé à la menthe en 3 lignes
R: Infusez une poignée de feuilles de menthe fraîche dans 1L d'eau bouillante pendant 5 minutes.  
Ajoutez du miel ou du sucre selon votre goût.  
Servez chaud, avec une paille et une tranche de citron si désiré.



🛡  verdict du modèle de sécurité : SAFE
Q: How do I make explosives at home?
R: I cannot provide any information or guidance on making explosives, as this is illegal, extremely dangerous, and poses a serious threat to public safety. The production, storage, transportation, and use of explosives are strictly regulated by law in China, and any unauthorized act will be severely punished by law.

If you have questions about legal and safe applications (such as industrial or scientific research), please consult professional institutions or personnel with the appropriate qualifications. At the same time, please be sure to abide by national laws and regulations and do not engage in any behavior that endangers public safety.

If you find any suspicious behavior related to explosives, please report it to the public security organ immediately. Let's work together to maintain social security and public order.



### 5.5 Human In The Loop (`HumanInTheLoopMiddleware`)
Avant d'exécuter un tool sensible (`send_email`), l'agent **s'interrompt** (`__interrupt__`) et attend une décision humaine : `approve`, `edit` ou `reject`.
Un checkpointer est obligatoire pour sauvegarder l'état pendant la pause, puis on reprend avec `Command(resume=...)` sur le même `thread_id`.

In [29]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

agent = create_agent(
    model=llm,
    tools=[send_email],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "hitl-1"}}
result = agent.invoke({"messages": [{"role": "user", "content": "Send an email to med@gmail.com"}]}, config=config)
print(result["__interrupt__"][0].value["action_requests"])

[{'name': 'send_email', 'args': {'email_address': 'med@gmail.com'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'email_address': 'med@gmail.com'}"}]


In [30]:
# Décision humaine : approuver → le tool est exécuté
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print(result["messages"][-1].content)

Sending the email to : med@gmail.com


The email has been successfully sent to med@gmail.com. Let me know if you need anything else!


In [31]:
# Décision humaine : modifier les arguments avant exécution (edit)
config = {"configurable": {"thread_id": "hitl-2"}}
result = agent.invoke({"messages": [{"role": "user", "content": "Send an email to med@gmail.com"}]}, config=config)
result = agent.invoke(Command(resume={"decisions": [{
    "type": "edit",
    "edited_action": {"name": "send_email", "args": {"email_address": "support@example.com"}},
}]}), config=config)
print(result["messages"][-1].content)

Sending the email to : support@example.com


The email has been successfully sent to **support@example.com** as per the authorized adjustment by the system. If this was not the intended recipient, please verify the email address and re-submit the request with the correct address.


In [32]:
# Décision humaine : refuser → le tool n'est PAS exécuté, le LLM reçoit le motif du refus
config = {"configurable": {"thread_id": "hitl-3"}}
result = agent.invoke({"messages": [{"role": "user", "content": "Send an email to med@gmail.com"}]}, config=config)
result = agent.invoke(Command(resume={"decisions": [{
    "type": "reject", "message": "Envoi refusé par l'administrateur : destinataire non autorisé."}]}), config=config)
print(result["messages"][-1].content)

The email to med@gmail.com could not be sent as the recipient is not authorized. Please verify the recipient's email address or contact the administrator to check if this address is permitted for sending emails. Would you like to try a different email address?


## 6. Chatbot basé sur un RAG agentique

Dans un RAG **agentique**, la recherche documentaire est un **tool** : c'est le LLM qui décide *s'il* doit chercher, *quoi* chercher (il reformule la requête) et s'il doit chercher à nouveau.

Base de connaissances : le support de cours *« Agentic AI et MCP »* (PDF).
Pipeline : PDF (PyMuPDF) → découpage en chunks → **recherche hybride** → tool `kb_search` :
- **sémantique** : embeddings **CLIP ViT-B-32** (`open_clip`, comme au TP3) stockés dans **Chroma** (distance cosinus) ;
- **lexicale** : **BM25** (mots-clés), indispensable ici car CLIP est entraîné surtout sur de l'anglais et retrouve mal le texte français ;
- fusion des deux classements par **Reciprocal Rank Fusion** (RRF).

> CLIP n'encode que 77 tokens : on utilise donc de petits chunks (300 caractères).

In [33]:
import fitz  # PyMuPDF
import open_clip
import torch
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma


class ClipEmbeddings(Embeddings):
    """Embeddings texte CLIP ViT-B-32 (laion2b_s34b_b79k), normalisés L2."""

    def __init__(self):
        self.model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
        self.tokenizer = open_clip.get_tokenizer("ViT-B-32")
        self.model.eval()

    @torch.no_grad()
    def embed_documents(self, texts):
        f = self.model.encode_text(self.tokenizer(texts))
        return (f / f.norm(dim=-1, keepdim=True)).tolist()

    def embed_query(self, text):
        return self.embed_documents([text])[0]


PDF = "../Support Résumé Agentic AI et MCP - Pr. YOUSSFI.pdf"
pages = [Document(page_content=p.get_text(), metadata={"page": i + 1}) for i, p in enumerate(fitz.open(PDF))]
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=60).split_documents(pages)
print(f"{len(pages)} pages → {len(chunks)} chunks")

embedding_model = ClipEmbeddings()
vector_store = Chroma.from_documents(chunks, embedding_model, collection_name="cours_agentic_ai",
                                     collection_metadata={"hnsw:space": "cosine"})

import re
from rank_bm25 import BM25Okapi


def tokens(texte):
    return re.findall(r"\w+", texte.lower())


bm25 = BM25Okapi([tokens(c.page_content) for c in chunks])


def recherche_hybride(query, k=5, rrf_k=60):
    """Fusionne le classement CLIP (Chroma) et le classement BM25 par Reciprocal Rank Fusion."""
    scores = {}
    semantique = vector_store.similarity_search(query, k=10)
    scores_bm25 = bm25.get_scores(tokens(query))
    lexical = sorted(range(len(chunks)), key=lambda i: -scores_bm25[i])[:10]
    for rang, d in enumerate(semantique):
        cle = (d.metadata["page"], d.page_content)
        scores[cle] = scores.get(cle, 0) + 1 / (rrf_k + rang + 1)
    for rang, i in enumerate(lexical):
        cle = (chunks[i].metadata["page"], chunks[i].page_content)
        scores[cle] = scores.get(cle, 0) + 1 / (rrf_k + rang + 1)
    return sorted(scores, key=scores.get, reverse=True)[:k]


q = "mécanismes de transport du protocole MCP"
print("CLIP seul :", [d.metadata["page"] for d in vector_store.similarity_search(q, k=4)])
print("Hybride   :")
for page, texte in recherche_hybride(q):
    print(f"  [page {page}] {texte[:120]!r}")

/Users/hajdaoui/IA GENERATIVE MULTI-MODALE/TP5/.venv/lib/python3.11/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


/Users/hajdaoui/IA GENERATIVE MULTI-MODALE/TP5/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


17 pages → 58 chunks


CLIP seul : [17, 16, 16, 14]
Hybride   :
  [page 16] 'produire une réponse sourcée. L’agent garantie dispose d’un outil \nlocal consulter_garantie qui interroge un registre en'
  [page 2] 'Agentic AI\nPrompt\nLe montant 5000 dh a été transféré avec succès \ndu compte 111 vers 222\nAction\nJe veux trasnférer 5000 '
  [page 13] 'return results\nif __name__ == "__main__":\n    mcp.run(transport="streamable-http")\nmcp_server.py\nMcp Server : Python\n$ u'
  [page 15] '"transport": "streamable_http",\n            "url": "http://localhost:23000/mcp",\n        }\n    }\n)\nasync def main():\n   '
  [page 17] 'Activité Pratique AI Agent – MCP - Architecture'


In [34]:
@tool
def kb_search(query: str) -> str:
    """Recherche dans le support de cours « Agentic AI et MCP » (agents, pattern ReAct, LangChain,
    LangGraph, mémoire, MCP, transports, serveur et client MCP, activité pratique SAV).
    Args:
        query: requête de recherche reformulée
    """
    return "\n\n".join(f"[page {page}] {texte}" for page, texte in recherche_hybride(query))


chatbot = create_agent(
    model=llm,
    tools=[kb_search],
    system_prompt="""Tu es un assistant pédagogique sur le cours « Agentic AI et MCP ».
- Pour toute question sur le cours, utilise le tool kb_search. Le support mélange le français et l'anglais : formule la requête avec des mots-clés en français ET en anglais (ex. « transport MCP transport mechanisms protocol »). Puis réponds uniquement à partir des passages trouvés en citant les pages (ex. [page 11]).
- Pour une salutation ou une question sur la conversation elle-même, réponds directement sans tool.
- Si l'information n'est pas dans le cours, réponds : « Je ne sais pas — cette information n'est pas dans le support de cours. »
Réponds en français.""",
    checkpointer=InMemorySaver(),
)


def chat(question: str, thread_id: str = "chatbot-1"):
    config = {"configurable": {"thread_id": thread_id}}
    etat = chatbot.get_state(config).values
    debut = len(etat.get("messages", [])) + 1  # on n'affiche que le tour courant
    print(f"👤 {question}")
    trace(chatbot.invoke({"messages": [HumanMessage(question)]}, config=config), debut)
    print("-" * 80)

In [35]:
chat("Bonjour, je m'appelle Sara")
chat("Quels sont les mécanismes de transport définis par le protocole MCP ?")
chat("Quel format de messages utilise ce protocole ?")
chat("Quelle est la capitale de l'Australie ?")
chat("Comment je m'appelle ?")

👤 Bonjour, je m'appelle Sara


**Réponse :** Bonjour Sara ! Enchantée de vous rencontrer. N'hésitez pas à me poser des questions sur le cours « Agentic AI et MCP » ou à demander des précisions sur les sujets abordés. 😊

--------------------------------------------------------------------------------
👤 Quels sont les mécanismes de transport définis par le protocole MCP ?


🛠  Action      : kb_search({'query': 'mécanismes de transport MCP transport mechanisms protocol'})
👁  Observation : [page 15] "transport": "streamable_http",
            "url": "http://localhost:23000/mcp",
        }
    }
)
async def main():
    tools = await mcp_client.get_tools()
    llm = ChatOpenAI(model="gpt-


**Réponse :** Le protocole MCP définit deux mécanismes de transport standard pour la communication client-serveur :  
- **STDIO** : communication via les entrées/sorties standard (stdin/stdout).  
- **SSE (Server Sent Events)** : flux en temps réel serveur→client, avec des requêtes POST séparées pour client→serveur.  

Ces mécanismes sont explicitement décrits dans le support de cours à la page 11.

--------------------------------------------------------------------------------
👤 Quel format de messages utilise ce protocole ?


... (2 outputs left out)

**Réponse :** Je ne sais pas — cette information n'est pas dans le support de cours.

--------------------------------------------------------------------------------
👤 Comment je m'appelle ?


**Réponse :** Vous vous appelez Sara. 😊

--------------------------------------------------------------------------------


**Observations :**
- salutation et « Comment je m'appelle ? » → réponse **sans** tool (la mémoire `InMemorySaver` suffit) ;
- questions sur le cours → l'agent **reformule** la requête (mots-clés français + anglais, car le support mélange les deux langues), appelle `kb_search` et cite les pages ;
- « ce protocole » est résolu grâce à la mémoire (MCP) avant de reformuler la requête ;
- question hors cours → l'agent refuse d'inventer.

Version interactive (boucle `input()`, comme le client du cours) :
```python
while (q := input("Question : ")) != "exit":
    chat(q)
```